# Arbeitspaket (AP) 3: Management & Nutzung Räumliche Daten

### Angaben Studierende(r) (fehlende Angaben ergänzen)

<table>
  <tr>
    <td>Vorname:</td>
    <td>Mats</td>
  </tr>
  <tr>
    <td>Nachname:</td>
    <td>Thiele</td>
  </tr>
  <tr>
    <td>Immatrikulationsnummer:</td>
    <td>25-194-721</td>
  </tr>
  <tr>
    <td>Modul:</td>
    <td>Data Science</td>
  </tr>
  <tr>
    <td>Prüfungsdatum / Raum / Zeit:</td>
    <td>06.10.2025 / Raum: MU O2.001 / 8:00 – 11:45</td>
  </tr>
  <tr>
    <td>Erlaubte Hilfsmittel:</td>
    <td>w.MA.XX.DS.25HS (Data Science)<br>Open Book, Eigener Computer, Internet-Zugang</td>
  </tr>
  <tr>
  <td>Nicht erlaubt:</td>
  <td>Nicht erlaubt ist der Einsatz beliebiger Formen von generativer KI (z.B. Copilot, ChatGPT) <br> sowie beliebige Formen von Kommunikation oder Kollaboration mit anderen Menschen.</td>
</tr>
</table>

## Bewertungskriterien

### <b style="color: gray;">(max. erreichbare Punkte: 40)</b>

<table>
  <thead>
    <tr>
      <th>Kategorie</th>
      <th>Beschreibung</th>
      <th>Punkteverteilung</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Code nicht lauffähig oder Ergebnisse nicht sinnvoll</td>
      <td>Der Code enthält Fehler, die verhindern, dass er ausgeführt werden kann (z.B. Syntaxfehler) oder es werden Ergebnisse ausgegeben, welche nicht zur Fragestellung passen.</td>
      <td>0 Punkte</td>
    </tr>
    <tr>
      <td>Code lauffähig, aber mit gravierenden Mängeln</td>
      <td>Der Code läuft, aber die Ergebnisse sind aufgrund wesentlicher Fehler unvollständig (z.B. fehlende Joins, gravierende Fehler in SQL-Abfragen). Nur geringer Fortschritt erkennbar.</td>
      <td>25% der max. erreichbaren Punkte</td>
    </tr>
    <tr>
      <td>Code lauffähig, aber mit mittleren Mängeln</td>
      <td>Der Code läuft und liefert teilweise korrekte Ergebnisse, aber es gibt grössere Fehler (z.B. fehlende Spalten, unvollständige SQL-Abfragen). Die Ergebnisse sind nachvollziehbar, aber unvollständig oder ungenau.</td>
      <td>50% der max. erreichbaren Punkte</td>
    </tr>
    <tr>
      <td>Code lauffähig, aber mit minimalen Mängeln</td>
      <td>Der Code läuft und liefert ein weitgehend korrektes Ergebnis, aber kleinere Fehler (z.B. falsche oder fehlende Sortierung, Rundung von Werten falsch) beeinträchtigen die Vollständigkeit des Ergebnisses.</td>
      <td>75% der max. erreichbaren Punkte</td>
    </tr>
    <tr>
      <td>Code lauffähig und korrekt</td>
      <td>Der Code läuft einwandfrei und liefert das korrekte Ergebnis ohne Mängel.</td>
      <td>100% der max. erreichbaren Punkte</td>
    </tr>
  </tbody>
</table>


#### <b>Python Libraries und Settings</b>

In [1]:
# Libraries
import os
import folium
import pandas as pd
import geopandas as gpd
from IPython.display import display, IFrame
from sqlalchemy import create_engine, text

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

print(os.getcwd())


/workspaces/python_postgresql_postgis/LNW


## <b>Vorbereitung (Hinweis: dieser Teil wird <u>nicht</u> bewertet)</b>

#### <b>1.) Starten Sie eine GitHub Codespaces Instanz auf Basis Ihres Forks des folgenden GitHub Repositories:</b>

##### GitHub-Repository: https://github.com/mario-gellrich-zhaw/python_postgresql_postgis

##### <span style="color: red;"><b>WICHTIG!!! Verwenden Sie eine GitHub Codespaces Instanz mit ausreichend Arbeitsspeicher (4core, 16GB RAM).</b></span>

<b>Hinweis:</b> 
- Im Unterricht wurden bereits sämtliche Installationen und Einstellungen inkl. der Registrierung des Datenbank Servers auf pgAdmin vorgenommen.
- Falls Sie die Codespaces-Instanz neu erstellen müssen, folgen Sie bitte den detaillierten Erklärungen auf der README-Seite des GitHub Repositories.

#### <b>2.) Erstellen und Testen Sie die Datenbankverbindung mit der 'osm_switzerland' Datenbank.</b>

In [2]:
# Set up Database Connection
user = "pgadmin"
password = "geheim"
host = "localhost"
port = "5432"
database = "osm_switzerland"

# Erstellen der Connection URL
db_connection_url = "postgresql://" + user + ":" + password +\
                    "@" + host + ":" + port + "/" + database

# Erstellen SQLAlchemy Engine
engine = create_engine(db_connection_url)

# Test der Connection
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT current_database();"))
        db_name = result.scalar()
        print("Aktuelle Datenbank:", db_name)
except Exception as e:
    print("Fehler bei der Verbindung:", e)

# Verbindung trennen
engine.dispose()

Aktuelle Datenbank: osm_switzerland


#### <b>3.) Legen Sie einen neuen Order 'maps' im Arbeitsverzeichnis an. In diesem werden Ihre generierten Karten abgespeichert.</b>

<b style="color:red;">Hinweis: Der Order 'maps' inklusive die erzeugten Karten muss als Teil des AP03 mit auf EXAM-Moodle abgegeben werden!</b>

In [3]:
# Anlegen eines Ordners 'maps'
if not os.path.exists('maps'):
    os.makedirs('maps')
    print("Ordner 'maps' wurde erstellt.")
else:
    print("Ordner 'maps' existiert bereits und wird nicht neu angelegt!")

Ordner 'maps' existiert bereits und wird nicht neu angelegt!


## <b>Aufgaben (Dieser Teil wird bewertet!)</b>
<b>Hinweise zu den folgenden Aufgabenstellungen:</b>
<ul>
  <li>In diesem Jupyter Notebook gibt es jeweils zwei Code-Zellen pro Aufgabe:</li>
  <ol>
    <li>Eine Codezelle mit Python-Code für die SQL-Abfrage.</li>
    <li>Eine Codezelle mit Python-Code für die Kartendarstellung der Ergebnisse der jeweiligen SQL-Abfrage.</li>
  </ol>
  <li>In den Codezellen für die SQL-Abfrage muss jeweils die SQL-Statement ergänzt werden.</li>
  <li>In den Codezellen für die Kartendarstellung muss nur dann der Python Code ergänzt werden, wenn in der Aufgabe danach gefragt wird.</li>
</ul>
<b style="color:red;">Beachten Sie, dass für die Punktevergabe auch die weiteren Anforderungen zu den Fragen unter 'Details zur Aufgabenstellung' herangezogen werden.</b>

#### <b>Aufgabe (1): Erstellen Sie eine Abfrage der Spitäler.</b>
<b>Details zur Aufgabenstellung:</b>
- Sie finden die benötigten Daten in der Tabelle 'planet_osm_polygon'.
- Stellen sie in der Ergebnistabelle die Spalten: osm_id, amenity sowie die transformierte Geometrie als Spalte geom dar.
- Tipps:
    - Die Geometry wird mit Hilfe der Funktion st_transform() transformiert, z.B.: *st_transform(p.way, 4326) AS geom*.
    - Die Daten sind mit dem key:value Paar amenity='hospital' in der OpenStreetMap Map-Feature Übersicht angegeben.
    - vgl: https://wiki.openstreetmap.org/wiki/Map_features

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [6]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# Ergänzen Sie die SQL-Abfrage, um die Aufgabe zu lösen
sql = """SELECT
            p.osm_id,
            p.amenity,
            st_transform(p.way, 4326) AS geom
        FROM
            planet_osm_polygon AS p
        WHERE p.amenity = 'hospital';"""

# Ergebnis in GeoDataFrame abspeichern
gdf_hospital = gpd.GeoDataFrame.from_postgis(sql, engine)

# Datenbankverbindung trennen
engine.dispose()

# Zeigen des GeoDataFrames
gdf_hospital.head(10)

,osm_id,amenity,geom
0,165847373,hospital,"POLYGON ((8.72864 47.51029, 8.72927 47.51012, ..."
1,-18241258,hospital,"POLYGON ((8.57208 47.35379, 8.57228 47.35284, ..."
2,-18241258,hospital,"POLYGON ((8.56919 47.3518, 8.56975 47.35114, 8..."
3,125667230,hospital,"POLYGON ((8.57592 47.3519, 8.57597 47.35179, 8..."
4,675191947,hospital,"POLYGON ((8.59648 47.34771, 8.5972 47.34769, 8..."
5,42156983,hospital,"POLYGON ((8.5172 47.41482, 8.51749 47.41474, 8..."
6,673609234,hospital,"POLYGON ((8.44987 47.47785, 8.44994 47.47757, ..."
7,248637584,hospital,"POLYGON ((8.68342 47.50718, 8.68342 47.5071, 8..."
8,856698475,hospital,"POLYGON ((8.56529 47.44926, 8.56576 47.44914, ..."
9,558245721,hospital,"POLYGON ((8.55653 47.38088, 8.55655 47.3808, 8..."


##### <b>Kartendarstellung Ergebnis (nur anpassen, falls in der Aufgabe danach gefragt wird)</b>

In [7]:
# Projektion definieren (WGS84)
if gdf_hospital.crs is None:
    gdf_hospital.set_crs(epsg=4326, inplace=True)
else:
    pass

# Latitude und Longitude für die Zentrierung der Karte ermitteln
centroids = gdf_hospital.geometry.centroid
lon = centroids.x.mean()
lat = centroids.y.mean()

# Initialisieren der Map
m = folium.Map(location=[lat, lon], 
               zoom_start=11, 
               tiles='CartoDBPositron')

# Map settings
folium.GeoJson(
    gdf_hospital,
    name='map'
).add_to(m)

folium.LayerControl().add_to(m)

# Save the map to an HTML file
m.save('maps/map_hospital.html')
print("Die Karte wurde unter 'maps/map_hospital.html' gespeichert.")

Die Karte wurde unter 'maps/map_hospital.html' gespeichert.


#### <b>Aufgabe (2) Erstellen Sie eine Abfrage der SBB Bahnhöfe.</b>
<b>Details zur Aufgabenstellung:</b>
- Verwenden Sie die Tabelle 'planet_osm_point'.
- Schränken Sie nach Bahnhöfen der SBB ein.
- Stellen Sie in der Ergebnistabelle die Spalten: osm_id, name, operator, public_transport und die transformierte Geometrie als Spalte geom dar.
- Tipp: Die Daten sind mit dem key:value Paar public_transport='station' in der OpenStreetMap Map-Feature Übersicht angegeben.
-  vgl: https://wiki.openstreetmap.org/wiki/Map_features

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [8]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# Ergänzen Sie die SQL-Abfrage, um die Aufgabe zu lösen
sql = """SELECT
            p.osm_id,
            p.name,
            p.operator,
            p.public_transport,
            st_transform(p.way, 4326) AS geom
        FROM
            planet_osm_point AS p
        WHERE p.public_transport = 'station'
            AND p.operator = 'SBB'
        ;"""

# Ergebnis in GeoDataFrame abspeichern
gdf_stations = gpd.GeoDataFrame.from_postgis(sql, engine)

# Datenbankverbindung trennen
engine.dispose()

# Zeigen des GeoDataFrames
gdf_stations.head(10)

,osm_id,name,operator,public_transport,geom
0,4574932116,Rüti,SBB,station,POINT (8.85436 47.26035)
1,2414238691,Tann-Dürnten,SBB,station,POINT (8.85245 47.26845)
2,2443333883,Steg,SBB,station,POINT (8.93238 47.35388)
3,2446055623,Wila,SBB,station,POINT (8.84854 47.41778)
4,25586596,Winterthur Wallrüti,SBB,station,POINT (8.76089 47.51672)
5,1310783650,Winterthur Seen,SBB,station,POINT (8.76675 47.48743)
6,1351877724,Schottikon,SBB,station,POINT (8.81174 47.5001)
7,1352668875,Rikon,SBB,station,POINT (8.79644 47.44501)
8,1241075021,Kemptthal,SBB,station,POINT (8.70581 47.45285)
9,1330704266,Winterthur Töss,SBB,station,POINT (8.70928 47.48977)


##### <b>Kartendarstellung Ergebnis (nur anpassen, falls in der Aufgabe danach gefragt wird)</b>

In [9]:
# Projektion definieren (WGS84)
if gdf_stations.crs is None:
    gdf_stations.set_crs(epsg=4326, inplace=True)
else:
    pass

# Latitude und Longitude für die Zentrierung der Karte ermitteln
centroids = gdf_stations.geometry.centroid
lon = centroids.x.mean()
lat = centroids.y.mean()

# Initialisieren der Map
m = folium.Map(location=[lat, lon], 
               zoom_start=10, 
               tiles='CartoDBPositron')

# Map settings with popup
folium.GeoJson(
    gdf_stations,
    name='map'
).add_to(m)

folium.LayerControl().add_to(m)

# Save the map to an HTML file
m.save('maps/map_stations.html')
print("Die Karte wurde unter 'maps/map_stations.html' gespeichert.")

Die Karte wurde unter 'maps/map_stations.html' gespeichert.


#### <b>Aufgabe (3): Erstellen Sie eine Abfrage aller Strassen, die als 'primary', 'secondary' oder 'tertiary' klassifiziert sind mit Infos zur Art des Fahrradwegs.</b>
<b>Details zur Aufgabenstellung:</b>
- Sie finden die benötigten Informationen in der Tabelle 'planet_osm_line'.
- Stellen Sie in der Ergebnistabelle die Spalten: osm_id, highway, bicycle und die transformierte Geometrie als Spalte geom dar.
- Schränken Sie die Abfrage auf alle Strassen ein, bei denen Information zur Art des Fahrradweges vorliegen.
- Tipp: Sie finden die Informationen zur Art der Fahrradwegs in der Spalte 'bicycle'.
-  vgl: https://wiki.openstreetmap.org/wiki/Map_features

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [10]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)

# Ergänzen Sie die SQL-Abfrage, um die Aufgabe zu lösen
sql = """SELECT
            p.osm_id,
            p.highway,
            p.bicycle,
            st_transform(p.way, 4326) AS geom
        FROM
            planet_osm_line AS p
        WHERE p.bicycle = 'yes'
            AND p.highway IN ('primary', 'secondary', 'tertiary')
        ;"""

# Ergebnis in GeoDataFrame abspeichern
gdf_streets = gpd.GeoDataFrame.from_postgis(sql, engine)

# Datenbankverbindung trennen
engine.dispose()

# Zeigen des GeoDataFrames
gdf_streets.head(10)

,osm_id,highway,bicycle,geom
0,25489279,tertiary,yes,"LINESTRING (8.76921 47.5013, 8.76934 47.50165)"
1,51723191,tertiary,yes,"LINESTRING (8.76885 47.50069, 8.76902 47.50094..."
2,25489280,tertiary,yes,"LINESTRING (8.76934 47.50165, 8.7694 47.50188,..."
3,1011393179,tertiary,yes,"LINESTRING (8.76797 47.49957, 8.76845 47.50019..."
4,113218780,tertiary,yes,"LINESTRING (8.76788 47.49946, 8.76797 47.49957)"
5,650589827,tertiary,yes,"LINESTRING (8.74647 47.46126, 8.7465 47.46127,..."
6,783865084,tertiary,yes,"LINESTRING (8.73715 47.44025, 8.73719 47.44032..."
7,1227404378,tertiary,yes,"LINESTRING (8.73642 47.43873, 8.73652 47.43894..."
8,783865091,tertiary,yes,"LINESTRING (8.73608 47.43811, 8.73619 47.43824..."
9,1227475814,tertiary,yes,"LINESTRING (8.7359 47.43788, 8.73576 47.43788,..."


##### <b>Kartendarstellung Ergebnis (nur anpassen, falls in der Aufgabe danach gefragt wird)</b>

In [11]:
# Projektion definieren (WGS84)
if gdf_streets.crs is None:
    gdf_streets.set_crs(epsg=4326, inplace=True)
else:
    pass

# Latitude und Longitude für die Zentrierung der Karte ermitteln
centroids = gdf_streets.geometry.centroid
lon = centroids.x.mean()
lat = centroids.y.mean()

# Initialisieren der Map
m = folium.Map(location=[lat, lon], 
               zoom_start=11, 
               tiles='CartoDBPositron')

# Map settings
folium.Choropleth(
    geo_data=gdf_streets,
    name='map',
    line_weight=3,
    line_color='darkred'
).add_to(m)

folium.LayerControl().add_to(m)

# Save the map to an HTML file
m.save('maps/map_streets.html')
print("Die Karte wurde unter 'maps/map_streets.html' gespeichert.")

Die Karte wurde unter 'maps/map_streets.html' gespeichert.


#### <b>Aufgabe (4): Erstellen Sie eine Abfrage aller Gebäude in der Bahnhofstrasse in Zürich, bei denen mindestens die Postleitzahl (PLZ) vorhanden ist.</b>
<b>Details zur Aufgabenstellung:</b>
- Verwenden Sie die Tabelle 'planet_osm_polygon'.
- Stellen Sie in der Ergebnistabelle Strassenname, Haunummer, PLZ, Gemeindename sowie die transformierte Geometrie als Spalte geom dar.
- Dabei ist nicht wichtig, dass alle Adress-Attribute für jedes Gebäude vorhanden sind, nur die PLZ muss immer vorhanden sein.
- Verwenden Sie für die Darstellung als Hintergrundkarte ein Satellitenbild (ESRIWorldImagery) als maptile.
- Tipp: Gebäude sind in der Spalte 'building' klassifiziert. Mit *WHERE building IS NOT NULL* können Sie Gebäude filtern.

<b style="color: gray;">(max. erreichbare Punkte: 4)</b>

In [12]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)  

# Ergänzen Sie die SQL-Abfrage, um die Aufgabe zu lösen
sql = """SELECT
            p.osm_id,
            p."addr:street",
            p."addr:housenumber",
            p."addr:postcode",
            p."addr:city",
            ST_Transform(p.way, 4326) AS geom
        FROM planet_osm_polygon p
        WHERE p.building IS NOT NULL
        AND p."addr:city" = 'Zürich'
        AND p."addr:street" = 'Bahnhofstrasse'
        AND p."addr:postcode" IS NOT NULL
        ;"""

# Ergebnis in GeoDataFrame abspeichern
gdf_buildings = gpd.GeoDataFrame.from_postgis(sql, engine)

# Datenbankverbindung trennen
engine.dispose()

# Zeigen des GeoDataFrames
gdf_buildings.head(10)

,osm_id,addr:street,addr:housenumber,addr:postcode,addr:city,geom
0,291657083,Bahnhofstrasse,92,8001,Zürich,"POLYGON ((8.53944 47.37577, 8.53957 47.37574, ..."
1,291221102,Bahnhofstrasse,84,8001,Zürich,"POLYGON ((8.53921 47.37532, 8.53937 47.37528, ..."
2,290914347,Bahnhofstrasse,78,8001,Zürich,"POLYGON ((8.53898 47.37488, 8.53917 47.37483, ..."
3,290914348,Bahnhofstrasse,80,8001,Zürich,"POLYGON ((8.53903 47.37497, 8.53922 47.37492, ..."
4,290914345,Bahnhofstrasse,76,8001,Zürich,"POLYGON ((8.53893 47.37477, 8.53913 47.37473, ..."
5,291657084,Bahnhofstrasse,94,8001,Zürich,"POLYGON ((8.53952 47.37592, 8.53965 47.37589, ..."
6,291657085,Bahnhofstrasse,98,8001,Zürich,"POLYGON ((8.53959 47.37605, 8.53978 47.37601, ..."
7,291657079,Bahnhofstrasse,100,8001,Zürich,"POLYGON ((8.53964 47.37614, 8.53975 47.37612, ..."
8,318818101,Bahnhofstrasse,83,8001,Zürich,"POLYGON ((8.53901 47.37611, 8.53906 47.37609, ..."
9,325862013,Bahnhofstrasse,77,8001,Zürich,"POLYGON ((8.53838 47.37504, 8.5387 47.37496, 8..."


##### <b>Kartendarstellung Ergebnis (nur anpassen, falls in der Aufgabe danach gefragt wird)</b>

In [13]:
# Projektion definieren (WGS84)
if gdf_buildings.crs is None:
    gdf_buildings.set_crs(epsg=4326, inplace=True)
else:
    pass

# Latitude und Longitude für die Zentrierung der Karte ermitteln
centroids = gdf_buildings.geometry.centroid
lon = centroids.x.mean()
lat = centroids.y.mean()

# Initialisieren der Map
m = folium.Map(location=[lat, lon], 
               zoom_start=16, 
               tiles='ESRIWorldImagery')

# Map settings
folium.Choropleth(
    geo_data=gdf_buildings,
    name='map',
    fill_color='orange'
).add_to(m)

folium.LayerControl().add_to(m)

# Save the map to an HTML file
m.save('maps/map_buildings.html')
print("Die Karte wurde unter 'maps/map_buildings.html' gespeichert.")

Die Karte wurde unter 'maps/map_buildings.html' gespeichert.


#### <b>Aufgabe (5): Erstellen Sie eine Abfrage des Greifensees, und generieren Sie einen Buffer um diesen See mit einer Breite von 500m.</b>
<b>Details zur Aufgabenstellung:</b>
- Sie finden die benötigten Informationen in der Tabelle 'planet_osm_polygon'.
- Stellen Sie in der Ergebnistabelle die Spalten: osm_id, name, natural sowie die transformierte Geometrie des Buffers als Spalte geom dar.
- Im Polygondatensatz gibt es unter dem Namen 'Greifensee' nicht nur einen Polygon. Filtern Sie in der Abfrage nur den See.
- Tipp: Zum Filtern des Sees können Sie die Spalte 'natural' verwenden.
-  vgl: https://wiki.openstreetmap.org/wiki/Map_features

<b style="color: gray;">(max. erreichbare Punkte: 8)</b>

In [14]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url) 

# Ergänzen Sie die SQL-Abfrage, um die Aufgabe zu lösen
sql = """SELECT
            p.osm_id,
            p.name,
            p.natural,
            ST_Transform(ST_Buffer(ST_Transform(p.way::geometry, 3857), 500), 4326) AS geom
        FROM
            planet_osm_polygon AS p
        WHERE
            p.name = 'Greifensee'
            AND p.natural = 'water'    
        ;"""

# Ergebnis in GeoDataFrame abspeichern
gdf_lake = gpd.GeoDataFrame.from_postgis(sql, engine)

# Datenbankverbindung trennen
engine.dispose()

# Zeigen des GeoDataFrames
gdf_lake

,osm_id,name,natural,geom
0,-1678346,Greifensee,water,"POLYGON ((8.6478 47.36811, 8.64782 47.3682, 8...."


##### <b>Kartendarstellung Ergebnis (nur anpassen, falls in der Aufgabe danach gefragt wird)</b>

In [15]:
# Projektion definieren (WGS84)
if gdf_lake.crs is None:
    gdf_lake.set_crs(epsg=4326, inplace=True)
else:
    pass

# Latitude und Longitude für die Zentrierung der Karte ermitteln
centroids = gdf_lake.geometry.centroid
lon = centroids.x.mean()
lat = centroids.y.mean()

# Initialisieren der Map
m = folium.Map(location=[lat, lon], 
               zoom_start=13, 
               tiles='ESRIWorldImagery')

# Map settings
folium.Choropleth(
    geo_data=gdf_lake,
    name='map',
    fill_color='greenyellow',
    fill_opacity=0.5,
    line_opacity=0.5
).add_to(m)

folium.LayerControl().add_to(m)

# Save the map to an HTML file
m.save('maps/map_lake.html')
print("Die Karte wurde unter 'maps/map_lake.html' gespeichert.")

Die Karte wurde unter 'maps/map_lake.html' gespeichert.


#### <b>Aufgabe (6): Erstellen Sie eine Abfrage der landwirtschaftlichen Nutzflächen in den Grenzen der Stadt Zürich.</b>
<b>Details zur Aufgabenstellung:</b>
- Sie finden die benötigten Daten in der Tabelle 'planet_osm_polygon'.
- Landwirtschaftliche Nutzflächen sollen in dieser Aufgabe sowohl 'farmland' als auch 'meadows' beinhalten.
- Selektieren Sie die landwirtschaftlichen Nutzflächen über die Gemeindefläche.
- Berechen Sie die Fläche der landwirtschaftlichen Nutzflächen und speichern diese in einer Spalte 'area_ha' ab.
- Stellen Sie in der Ergebnistabelle die Spalten: osm_id, landuse, area_ha sowie die transformierte Geometrie als geom dar.
- Wählen Sie eine Satelliten Karte von ESRI als Hintergrundkarte (maptile).
- Ordnen Sie die Ergebnistabelle nach area_ha absteigend.
- Tipp: Im Unterricht wurde eine ähnliche Abfrage mit Waldflächen (forest) behandelt.
- vgl: https://wiki.openstreetmap.org/wiki/Map_features

<b style="color: gray;">(max. erreichbare Punkte: 8)</b>

In [16]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)  

# Ergänzen Sie die SQL-Abfrage, um die Aufgabe zu lösen
sql = """SELECT
            p.osm_id,
            p.landuse,
            ST_Area(
                ST_Transform(
                    ST_Intersection(p.way, z.way),
                    32632
                )
            ) / 10000 AS area_ha,
            ST_Transform(
                ST_Intersection(p.way, z.way),
                4326
            ) AS geom
        FROM public.planet_osm_polygon AS p
        JOIN public.planet_osm_polygon AS z
        ON ST_Intersects(z.way, p.way)
        WHERE
            z.admin_level = '8'
            AND z.name = 'Zürich'
            AND z.boundary = 'administrative'
            AND p.landuse IN ('farmland', 'meadows')
        ORDER BY area_ha DESC;
        ;"""

# Ergebnis in GeoDataFrame abspeichern
gdf_agriculture = gpd.GeoDataFrame.from_postgis(sql, engine)

# Datenbankverbindung trennen
engine.dispose()

# Zeigen des GeoDataFrames
gdf_agriculture.head(10)

,osm_id,landuse,area_ha,geom
0,123222208,farmland,19.669290,"POLYGON ((8.58497 47.38446, 8.58561 47.38499, ..."
1,1419414136,farmland,14.229945,"POLYGON ((8.51675 47.42279, 8.5169 47.42321, 8..."
2,123083907,farmland,13.380850,"POLYGON ((8.5702 47.40221, 8.57022 47.40225, 8..."
3,37201232,farmland,13.357745,"POLYGON ((8.50796 47.40519, 8.50802 47.40525, ..."
4,425913586,farmland,12.912359,"POLYGON ((8.52453 47.41736, 8.52613 47.41781, ..."
5,48565300,farmland,12.572642,"POLYGON ((8.59669 47.3656, 8.59734 47.36564, 8..."
6,256312662,farmland,12.355890,"POLYGON ((8.52554 47.42685, 8.5258 47.42743, 8..."
7,123222207,farmland,11.884524,"POLYGON ((8.59151 47.38248, 8.59147 47.38246, ..."
8,37208273,farmland,11.556099,"POLYGON ((8.49861 47.40844, 8.49889 47.40839, ..."
9,258364953,farmland,10.811152,"POLYGON ((8.49861 47.42946, 8.49866 47.42958, ..."


##### <b>Kartendarstellung Ergebnis (nur anpassen, falls in der Aufgabe danach gefragt wird)</b>

In [17]:
# Projektion definieren (WGS84)
if gdf_agriculture.crs is None:
    gdf_agriculture.set_crs(epsg=4326, inplace=True)
else:
    pass

# Latitude und Longitude für die Zentrierung der Karte ermitteln
centroids = gdf_agriculture.geometry.centroid
lon = centroids.x.mean()
lat = centroids.y.mean()

# Initialisieren der Map
m = folium.Map(location=[lat, lon], 
               zoom_start=13, 
               tiles='EsriWorldImagery')

# Map settings
folium.GeoJson(
    gdf_agriculture,
    name='map',
).add_to(m)

folium.LayerControl().add_to(m)

# Save the map to an HTML file
m.save('maps/map_farmland_meadow.html')
print("Die Karte wurde unter 'maps/map_farmland_meadow.html' gespeichert.")

Die Karte wurde unter 'maps/map_farmland_meadow.html' gespeichert.


#### <b>Aufgabe (7): Erstellen Sie eine Abfrage der Hotels in einem Radius von 5km um den Flughafen Zürich.</b>
<b>Details zur Aufgabenstellung:</b>
- Verwenden Sie die Tabelle 'planet_osm_point'.
- Berechnen Sie in der Abfrage die Distanz jedes Hotels zum Flughafen in Metern als Spalte 'distance_meters'.
- Stellen Sie in der Ergebnistabelle die Spalten: osm_id, tourism, name, distance_meters, sowie die transformierte Geometrie als geom dar.
- Wählen Sie eine Satelliten Karte von ESRI als Hintergrundkarte (maptile).
- Integrieren Sie in die Kartendarstellung den Namen (Spalte 'name') der Hotels als Popup.
- Tipp: Hotels sind mit dem key:value Paar tourism='hotel' in der OpenStreetMap Map-Feature Übersicht angegeben.
- vgl: https://wiki.openstreetmap.org/wiki/Map_features

<b style="color: gray;">(max. erreichbare Punkte: 8)</b>

In [18]:
# Engine für Datenbankverbindung erstellen
engine = create_engine(db_connection_url)  

# Ergänzen Sie die SQL-Abfrage, um die Aufgabe zu lösen
sql = """SELECT
            p.osm_id,
            p.tourism,
            p.name,
            ST_Distance(
                ST_Transform(p.way, 4326)::geography,
                ST_SetSRID(ST_MakePoint(8.5619, 47.4501), 4326)::geography
            ) AS distance_meters,
            ST_TRANSFORM(p.way, 4326) AS geom
        FROM
            planet_osm_point AS p
        WHERE
            p.tourism = 'hotel'
            AND ST_DWithin(
                ST_Transform(p.way, 4326)::geography,
                ST_SetSRID(ST_MakePoint(8.5619, 47.4501), 4326)::geography,
                5000
            )
        ORDER BY distance_meters;"""

# Ergebnis in GeoDataFrame abspeichern
gdf_hotels = gpd.GeoDataFrame.from_postgis(sql, engine)

# Datenbankverbindung trennen
engine.dispose()

# Zeigen des GeoDataFrames
gdf_hotels.head(10)


,osm_id,tourism,name,distance_meters,geom
0,8607233055,hotel,Transit Hotel & Dayrooms,139.212658,POINT (8.56018 47.45056)
1,9553976171,hotel,Capsule Hotel Alpine Garden,223.837861,POINT (8.5627 47.45204)
2,3086649239,hotel,Dorint Airport-Hotel Zürich,1295.666868,POINT (8.56559 47.43872)
3,3783549518,hotel,STAY@Zurich Airport,1302.470792,POINT (8.56747 47.43901)
4,6286669885,hotel,Welcome Inn,1489.478369,POINT (8.5816 47.44914)
5,12072254954,hotel,Sleepz,1576.398704,POINT (8.58278 47.44947)
6,4269704591,hotel,Apart-Hotel Zürich by Hilton,1586.150086,POINT (8.57373 47.43831)
7,291687394,hotel,Radisson Hotel Zurich Airport,1643.831636,POINT (8.54014 47.45099)
8,6286669886,hotel,Fly away,1686.032683,POINT (8.58426 47.45007)
9,1060256103,hotel,Holiday Inn Express Zürich Airport,1876.297202,POINT (8.53715 47.45186)


In [19]:
# Projektion definieren (WGS84)
if gdf_hotels.crs is None:
    gdf_hotels.set_crs(epsg=4326, inplace=True)
else:
    pass

# Latitude und Longitude für die Zentrierung der Karte ermitteln
centroids = gdf_hotels.geometry.centroid
lon = centroids.x.mean()
lat = centroids.y.mean()

# Initialisieren der Map
m = folium.Map(location=[lat, lon], 
               zoom_start=13, 
               tiles='EsriWorldImagery')

# Map settings
folium.GeoJson(
    gdf_hotels,
    name='map',
    popup=folium.GeoJsonPopup(fields=['name'])
).add_to(m)

folium.LayerControl().add_to(m)

# Save the map to an HTML file
m.save('maps/map_hotels.html')
print("Die Karte wurde unter 'maps/map_hotels.html' gespeichert.")

Die Karte wurde unter 'maps/map_hotels.html' gespeichert.


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [20]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1030-azure
Datetime: 2025-10-06 09:55:30
Python Version: 3.12.1
IP Address: 127.0.0.1
-----------------------------------
